# Lesson 03 - Agentiske designmønstre

I denne lektion udforsker vi tre grundlæggende designmønstre til opbygning af effektive AI-agenter:

1. **Klare agentinstruktioner** — Udformning af præcise, rolledefinerende prompts, der guider agentens adfærd  
2. **Struktureret output med Pydantic-modeller** — Sikring af, at agenter returnerer forudsigelige, validerede data  
3. **Agenter med enkelt ansvar** — Design af fokuserede agenter, der hver især gør én ting godt

Vi anvender hvert mønster på et **rejsemålsanbefalingsscenarie**, hvor vi trinvis bygger et system, der kan foreslå destinationer, tjekke tilgængelighed og håndtere logistik.


## Opsætning


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Mønster 1: Klare agentinstruktioner

Det mest effektive mønster er også det simpleste: at skrive klare, detaljerede instruktioner til din agent.

Gode instruktioner definerer:
- **Hvem** agenten er (persona og tone)
- **Hvad** den skal gøre (trin-for-trin ansvarsområder)
- **Hvordan** den skal opføre sig (begrænsninger og stil)

Nedenfor skaber vi en rejsekonsulent-agent med eksplicitte instruktioner, der former hvert svar, den producerer.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Mønster 2: Struktureret output med Pydantic-modeller

Fri tekst er nyttig til samtaler, men systemer nedstrøms har brug for strukturerede data.  
Ved at kombinere **Pydantic-modeller** med en **værktøjsfunktion** kan vi:

- Definere et præcist skema for agentens output  
- Validere svar automatisk  
- Integrere agentresultater i applikationslogik pålideligt  

Vi introducerer også et værktøj, der returnerer destinationens detaljer, så agenten forankrer sine anbefalinger i virkelige data.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Mønster 3: Agenter med enkelt ansvar

Komplekse opgaver har fordel af at dele arbejdet ud på flere fokuserede agenter, hver med et enkelt ansvar:

- En **Destinationsekspert**, der kender til steder og tilgængelighed
- En **Logistikplanlægger**, der håndterer fly, hoteller og rejseplaner

Dette afspejler softwareingeniørprincippet om *adskillelse af bekymringer* — hver agent er lettere at teste, vedligeholde og forbedre uafhængigt.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Resumé

I denne lektion anvendte vi tre agentiske designmønstre på et rejseanbefalingsscenarie:

| Mønster | Hovedidé | Fordel |
|---|---|---|
| **Klare instruktioner** | Definer persona, ansvar og begrænsninger på forhånd | Konsistent, brandtilpasset agentadfærd |
| **Struktureret output** | Brug Pydantic-modeller som svarformat | Validerede, maskinlæsbare resultater |
| **Single Responsibility** | Giv hver agent én fokuseret opgave | Nemmere at teste, vedligeholde og sammensætte |

Disse mønstre sammensætter naturligt — du kan kombinere klare instruktioner med struktureret output indenfor en single-responsibility agent for at bygge robuste, produktionsklare systemer.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, bedes du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det oprindelige dokument på dets modersmål bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for eventuelle misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
